# FastAPI — Complete Practice Notebook
### For GenAI / AI Backend Engineers

---

| Module | Topic |
|---|---|
| 1 | FastAPI Fundamentals — ASGI, App Object, OpenAPI |
| 2 | HTTP Methods |
| 3 | Request Handling — Path, Query, Body, Headers, Files |
| 4 | Pydantic in FastAPI — Validators, Computed, Response Models |
| 5 | Dependency Injection — Depends(), Chaining, LLM Clients |
| 6 | Response Types — JSON, Streaming, File, HTML |
| 7 | Async Programming — Sync vs Async, Generators, Pitfalls |
| 8 | Middleware — Logging, CORS, Rate Limiting |
| 9 | Error Handling — HTTPException, Global Handlers |
| 10 | Lifespan Management — Startup, Shutdown, Resource Init |
| 11 | Background Tasks |
| 12 | API Architecture — Router, Versioning, Modular Design |
| 13 | Database Integration — SQLAlchemy, Async, Repository Pattern |
| 14 | Auth & Authorization — OAuth2, JWT, RBAC, API Keys |
| 15 | OpenAPI & Docs |
| 16 | Streaming APIs — SSE, Token Streaming (Critical for GenAI) |
| 17 | WebSockets — Real-Time AI Responses |
| 18 | Caching — Redis, LLM Response Cache |
| 19 | Observability — Structured Logging, OpenTelemetry |
| 20 | Testing — TestClient, Dependency Overrides, Mock LLMs |
| 21 | FastAPI for GenAI — RAG, Agents, MCP Servers, Streaming LLMs |
| 22 | Deployment — Docker, Health Checks, Gunicorn |

---
## Install

In [ ]:
!pip install fastapi uvicorn[standard] python-multipart python-jose[cryptography] passlib[bcrypt] sqlalchemy aiosqlite httpx pytest redis nest-asyncio

---
## Imports

In [ ]:
# Core FastAPI
from fastapi import FastAPI, APIRouter, Depends, HTTPException, status
from fastapi import Header, Cookie, Form, UploadFile, File, BackgroundTasks, Request, Response
from fastapi.responses import JSONResponse, HTMLResponse, PlainTextResponse, RedirectResponse
from fastapi.responses import FileResponse, StreamingResponse
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm, APIKeyHeader
from fastapi.testclient import TestClient

# Pydantic
from pydantic import BaseModel, Field, field_validator, computed_field, ConfigDict
from typing import Optional, List, Dict, Any, AsyncGenerator

# Auth
from jose import JWTError, jwt
from passlib.context import CryptContext
from datetime import datetime, timedelta

# Async + Utilities
import asyncio
import json
import time
import uuid
import nest_asyncio
nest_asyncio.apply()   # allows asyncio.run() inside Jupyter

print("All imports OK")

---
## Module 1 — FastAPI Fundamentals

### What is FastAPI?
FastAPI is a modern, high-performance Python web framework for building APIs with Python type hints.

| | Flask | Django | FastAPI |
|---|---|---|---|
| Speed | Medium | Slow | Very Fast |
| Async | No (native) | No (native) | Yes (native) |
| Auto docs | No | No | Yes (Swagger + ReDoc) |
| Type validation | No | No | Yes (Pydantic) |
| Learning curve | Low | High | Low-Medium |

### ASGI vs WSGI
- **WSGI** (Web Server Gateway Interface) — synchronous, one request at a time per worker. Used by Flask, Django.
- **ASGI** (Asynchronous Server Gateway Interface) — async-capable, handles thousands of concurrent connections. Used by FastAPI, Starlette.
- FastAPI runs on **Uvicorn** (ASGI server)

### FastAPI App Object
```python
app = FastAPI(
    title="My API",
    description="...",
    version="1.0.0",
    docs_url="/docs",      # Swagger UI
    redoc_url="/redoc"    # ReDoc
)
```

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient

app = FastAPI(
    title="My Learning API",
    description="FastAPI practice notebook — built for GenAI engineers",
    version="1.0.0",
    contact={"name": "Shriman", "email": "shriman@example.com"},
    license_info={"name": "MIT"}
)

@app.get("/", tags=["Root"])
def root():
    return {"message": "FastAPI is running!", "version": "1.0.0"}

@app.get("/info", tags=["Root"])
def info():
    return {
        "framework": "FastAPI",
        "server": "Uvicorn (ASGI)",
        "docs": "/docs"
    }

client = TestClient(app)
print(client.get("/").json())
print(client.get("/info").json())

---
## Module 2 — HTTP Methods

| Method | Purpose | Has Body? |
|---|---|---|
| `GET` | Retrieve a resource | No |
| `POST` | Create a resource | Yes |
| `PUT` | Replace a resource completely | Yes |
| `PATCH` | Partially update a resource | Yes |
| `DELETE` | Delete a resource | Optional |
| `HEAD` | Like GET but returns only headers | No |
| `OPTIONS` | Describe what methods are allowed | No |

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()

class Item(BaseModel):
    name: str
    price: float

db: Dict[int, dict] = {}

@app.get("/items/{item_id}")
def get_item(item_id: int):
    return db.get(item_id, {"error": "not found"})

@app.post("/items", status_code=201)
def create_item(item: Item):
    item_id = len(db) + 1
    db[item_id] = item.model_dump()
    return {"id": item_id, **item.model_dump()}

@app.put("/items/{item_id}")
def replace_item(item_id: int, item: Item):
    db[item_id] = item.model_dump()
    return {"id": item_id, **item.model_dump()}

@app.patch("/items/{item_id}")
def update_item(item_id: int, updates: dict):
    if item_id in db:
        db[item_id].update(updates)
    return db.get(item_id)

@app.delete("/items/{item_id}", status_code=204)
def delete_item(item_id: int):
    db.pop(item_id, None)

client = TestClient(app)

# POST — create
r = client.post("/items", json={"name": "Laptop", "price": 999.0})
print("POST:", r.json())

# GET
print("GET: ", client.get("/items/1").json())

# PATCH
client.patch("/items/1", json={"price": 899.0})
print("PATCH:", client.get("/items/1").json())

# DELETE
print("DELETE status:", client.delete("/items/1").status_code)

---
## Module 3 — Request Handling

### 3.1 Path Parameters
Embedded in the URL: `/users/{user_id}`  
FastAPI automatically validates and casts the type.

### 3.2 Query Parameters
Appended to URL: `/search?q=llm&limit=10`  
Defined as function params NOT in the path.

### 3.3 Request Body
JSON payload sent in POST/PUT/PATCH requests via a Pydantic model.

### 3.4 Headers
Meta-information: `Authorization`, `Content-Type`, `X-Request-ID`

### 3.5 File Uploads
`UploadFile` — async file object with `.filename`, `.content_type`, `.read()`

In [ ]:
from fastapi import FastAPI, Path, Query
from fastapi.testclient import TestClient
from typing import Optional

app = FastAPI()

# --- Path Parameters ---
@app.get("/users/{user_id}")
def get_user(
    user_id: int = Path(ge=1, description="User ID must be positive")
):
    return {"user_id": user_id, "name": f"User {user_id}"}

@app.get("/files/{file_path:path}")   # :path captures slashes too
def get_file(file_path: str):
    return {"path": file_path}

# --- Query Parameters ---
@app.get("/search")
def search(
    q: str,                             # required
    limit: int = Query(default=10, ge=1, le=100),
    offset: int = 0,
    published: Optional[bool] = None
):
    return {"query": q, "limit": limit, "offset": offset, "published": published}

client = TestClient(app)
print("Path param:  ", client.get("/users/5").json())
print("Path file:   ", client.get("/files/docs/guide/intro.md").json())
print("Query params:", client.get("/search?q=llm&limit=5&published=true").json())

In [ ]:
from fastapi import FastAPI, Header, Cookie
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import Optional

app = FastAPI()

class UserCreate(BaseModel):
    username: str
    email: str
    age: int
    address: Optional[dict] = None   # nested object

# --- Request Body ---
@app.post("/users")
def create_user(user: UserCreate):
    return {"created": True, "user": user.model_dump()}

# --- Headers ---
@app.get("/protected")
def protected(
    authorization: Optional[str] = Header(default=None),
    x_request_id: Optional[str] = Header(default=None)
):
    return {"auth": authorization, "request_id": x_request_id}

# --- Cookies ---
@app.get("/whoami")
def whoami(session_token: Optional[str] = Cookie(default=None)):
    return {"session_token": session_token}

client = TestClient(app)

# Body
r = client.post("/users", json={
    "username": "shriman", "email": "s@example.com", "age": 25,
    "address": {"city": "Bangalore"}
})
print("Body:   ", r.json())

# Headers
r = client.get("/protected", headers={"Authorization": "Bearer token123", "X-Request-ID": "abc"})
print("Headers:", r.json())

# Cookies
r = client.get("/whoami", cookies={"session_token": "sess_xyz"})
print("Cookie: ", r.json())

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form
from fastapi.testclient import TestClient
from typing import List
import io

app = FastAPI()

# Single file upload
@app.post("/upload")
async def upload_file(file: UploadFile = File(...)):
    content = await file.read()
    return {
        "filename": file.filename,
        "content_type": file.content_type,
        "size_bytes": len(content)
    }

# File + form fields together
@app.post("/upload-with-meta")
async def upload_with_meta(
    file: UploadFile = File(...),
    description: str = Form(...),
    tags: str = Form(default="")
):
    content = await file.read()
    return {
        "filename": file.filename,
        "description": description,
        "tags": tags.split(","),
        "size": len(content)
    }

# Multiple files
@app.post("/upload-multiple")
async def upload_multiple(files: List[UploadFile] = File(...)):
    results = []
    for f in files:
        content = await f.read()
        results.append({"name": f.filename, "size": len(content)})
    return {"files": results, "total": len(results)}

client = TestClient(app)

# Test single upload
fake_file = io.BytesIO(b"Hello, this is file content")
r = client.post("/upload", files={"file": ("test.txt", fake_file, "text/plain")})
print("Single file:", r.json())

# Test with form data
fake_file2 = io.BytesIO(b"PDF content here")
r = client.post("/upload-with-meta",
    files={"file": ("doc.pdf", fake_file2, "application/pdf")},
    data={"description": "Quarterly report", "tags": "finance,q1"}
)
print("With meta:  ", r.json())

---
## Module 4 — Pydantic in FastAPI

We covered Pydantic basics in `Pydantic_Practice.ipynb`. Here we focus on FastAPI-specific Pydantic features:

- **Custom Validators** — `@field_validator` for custom logic
- **Computed Fields** — `@computed_field` for derived values
- **Response Models** — `response_model=` to control what gets sent back to client
- The `response_model` strips sensitive fields (like passwords) from the response automatically

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, field_validator, computed_field
from typing import Optional

app = FastAPI()

class UserCreate(BaseModel):
    username: str = Field(min_length=3)
    email: str
    password: str = Field(min_length=8)
    age: int = Field(ge=18)

    @field_validator("email")
    @classmethod
    def email_must_have_at(cls, v):
        if "@" not in v:
            raise ValueError("Invalid email address")
        return v.lower()   # normalize to lowercase

    @field_validator("username")
    @classmethod
    def no_spaces_in_username(cls, v):
        if " " in v:
            raise ValueError("Username cannot contain spaces")
        return v.lower()

    @computed_field
    @property
    def domain(self) -> str:
        return self.email.split("@")[-1]

class UserResponse(BaseModel):   # Response model — NO password field!
    username: str
    email: str
    age: int
    domain: str

@app.post("/register", response_model=UserResponse)
def register(user: UserCreate):
    return user   # password is automatically stripped by response_model

client = TestClient(app)

from pydantic import ValidationError

# Valid user
r = client.post("/register", json={
    "username": "Shriman_01",
    "email": "Shriman@EXAMPLE.COM",
    "password": "secret123",
    "age": 25
})
print("Response (no password!):", r.json())

# Invalid email
r = client.post("/register", json={
    "username": "alice",
    "email": "not-an-email",
    "password": "secret123",
    "age": 22
})
print("Validation error:", r.status_code, r.json()["detail"][0]["msg"])

---
## Module 5 — Dependency Injection

`Depends()` is FastAPI's DI system. It lets you:
- Share logic across routes (auth checks, DB sessions, config)
- Compose complex dependencies by chaining
- Override in tests easily

```python
from fastapi import Depends

def get_db():          # dependency function
    db = connect()
    yield db           # yield = cleanup happens after request
    db.close()

@app.get("/items")
def get_items(db = Depends(get_db)):   # inject
    ...
```

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header
from fastapi.testclient import TestClient
from typing import Optional

app = FastAPI()

# --- Basic dependency ---
def get_settings():
    return {"app_name": "MyAPI", "debug": True, "max_results": 100}

@app.get("/settings")
def read_settings(settings: dict = Depends(get_settings)):
    return settings

# --- Auth dependency ---
FAKE_TOKENS = {"token-alice": "alice", "token-bob": "bob"}

def get_current_user(authorization: Optional[str] = Header(default=None)):
    if not authorization or not authorization.startswith("Bearer "):
        raise HTTPException(status_code=401, detail="Missing token")
    token = authorization.split(" ")[1]
    user = FAKE_TOKENS.get(token)
    if not user:
        raise HTTPException(status_code=401, detail="Invalid token")
    return user

@app.get("/me")
def get_me(user: str = Depends(get_current_user)):
    return {"user": user}

# --- Chained dependency ---
def require_admin(user: str = Depends(get_current_user)):
    if user != "alice":
        raise HTTPException(status_code=403, detail="Admins only")
    return user

@app.delete("/admin/nuke")
def admin_action(admin: str = Depends(require_admin)):
    return {"action": "nuked", "by": admin}

client = TestClient(app)
print("Settings:", client.get("/settings").json())
print("Me (alice):", client.get("/me", headers={"Authorization": "Bearer token-alice"}).json())
print("Unauth:    ", client.get("/me").status_code)
print("Bob→admin: ", client.delete("/admin/nuke", headers={"Authorization": "Bearer token-bob"}).status_code)

In [ ]:
# LLM Client as a Dependency
# In production, replace MockLLMClient with actual anthropic.Anthropic() or openai.OpenAI()

from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient
from pydantic import BaseModel

class MockLLMClient:
    """Replace with: anthropic.Anthropic() or openai.OpenAI()"""
    def __init__(self, model: str = "claude-sonnet-4-6"):
        self.model = model

    def complete(self, prompt: str) -> str:
        return f"[Mock LLM response to: {prompt[:30]}...]"

# Dependency — singleton pattern via lru_cache or module-level
_llm_client = None

def get_llm_client() -> MockLLMClient:
    global _llm_client
    if _llm_client is None:
        _llm_client = MockLLMClient(model="claude-sonnet-4-6")
    return _llm_client

app = FastAPI()

class PromptRequest(BaseModel):
    prompt: str
    max_tokens: int = 1000

@app.post("/chat")
def chat(
    req: PromptRequest,
    llm: MockLLMClient = Depends(get_llm_client)
):
    response = llm.complete(req.prompt)
    return {"model": llm.model, "response": response}

client = TestClient(app)
r = client.post("/chat", json={"prompt": "Explain what is RAG in 3 sentences."})
print(r.json())

---
## Module 6 — Response Types

| Class | Use Case |
|---|---|
| `JSONResponse` | Default — returns JSON |
| `HTMLResponse` | Return HTML pages |
| `PlainTextResponse` | Plain text |
| `FileResponse` | Download a file |
| `RedirectResponse` | Redirect to another URL |
| `StreamingResponse` | Stream large data, LLM tokens, files |

`StreamingResponse` is **critical for GenAI** — it lets you stream tokens as they're generated.

In [ ]:
from fastapi import FastAPI
from fastapi.responses import JSONResponse, HTMLResponse, PlainTextResponse, RedirectResponse
from fastapi.testclient import TestClient

app = FastAPI()

@app.get("/json")
def json_resp():
    return JSONResponse(content={"hello": "world"}, status_code=200)

@app.get("/html", response_class=HTMLResponse)
def html_resp():
    return "<h1>Hello from FastAPI!</h1><p>This is HTML.</p>"

@app.get("/text", response_class=PlainTextResponse)
def text_resp():
    return "Just plain text. No JSON."

@app.get("/old-url")
def redirect():
    return RedirectResponse(url="/json", status_code=301)

@app.get("/custom-headers")
def custom():
    return JSONResponse(
        content={"status": "ok"},
        headers={"X-Custom-Header": "my-value", "Cache-Control": "no-cache"}
    )

client = TestClient(app, follow_redirects=False)
print("JSON:   ", client.get("/json").json())
print("HTML:   ", client.get("/html").text[:40])
print("Text:   ", client.get("/text").text)
print("Redir:  ", client.get("/old-url").status_code, client.get("/old-url").headers.get("location"))
print("Headers:", dict(client.get("/custom-headers").headers))

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
import asyncio
import time

app = FastAPI()

# Sync generator streaming
@app.get("/stream/sync")
def stream_sync():
    def data_generator():
        for i in range(5):
            yield f"chunk {i}\n"
    return StreamingResponse(data_generator(), media_type="text/plain")

# Async generator streaming (preferred for I/O bound work)
@app.get("/stream/async")
async def stream_async():
    async def async_generator():
        for i in range(5):
            await asyncio.sleep(0)   # yield control
            yield f"token_{i} "
    return StreamingResponse(async_generator(), media_type="text/plain")

# Large JSON streaming
@app.get("/stream/json-lines")
async def stream_jsonlines():
    async def generate():
        import json
        for i in range(5):
            row = {"id": i, "value": f"item-{i}"}
            yield json.dumps(row) + "\n"
    return StreamingResponse(generate(), media_type="application/x-ndjson")

client = TestClient(app)
print("Sync stream: ", client.get("/stream/sync").text)
print("Async stream:", client.get("/stream/async").text)
print("NDJSON:      ", client.get("/stream/json-lines").text)

---
## Module 7 — Async Programming

### sync vs async routes
```python
@app.get("/sync")         # runs in threadpool — safe for blocking I/O
def sync_route(): ...

@app.get("/async")        # runs in event loop — never block here!
async def async_route(): ...
```

### Rules
- Use `async def` when you use `await` (calling async libraries: httpx, asyncpg, motor)
- Use `def` (sync) for CPU-heavy or blocking code — FastAPI runs it in a thread automatically
- **Never** call `time.sleep()` inside `async def` → use `await asyncio.sleep()`
- **Never** call blocking I/O inside `async def` → use async equivalents

### Common Pitfalls
| Wrong | Right |
|---|---|
| `time.sleep(1)` in async | `await asyncio.sleep(1)` |
| `requests.get(url)` in async | `await httpx.AsyncClient().get(url)` |
| Heavy CPU in async | Run in `asyncio.run_in_executor()` |

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
import asyncio
import time
import concurrent.futures

app = FastAPI()

# Sync route — FastAPI runs this in a threadpool automatically
@app.get("/sync-task")
def sync_task():
    time.sleep(0.01)   # blocking — OK in sync route
    return {"type": "sync", "done": True}

# Async route — event loop based
@app.get("/async-task")
async def async_task():
    await asyncio.sleep(0.01)   # non-blocking
    return {"type": "async", "done": True}

# Async generator — for streaming
@app.get("/async-gen")
async def async_gen():
    async def generate():
        for word in ["Hello", " ", "from", " ", "async", " ", "FastAPI"]:
            await asyncio.sleep(0)
            yield word
    from fastapi.responses import StreamingResponse
    return StreamingResponse(generate(), media_type="text/plain")

# CPU-bound work in async — use executor to not block event loop
@app.get("/cpu-task")
async def cpu_task():
    loop = asyncio.get_event_loop()
    def heavy_computation():
        return sum(range(1_000_000))
    result = await loop.run_in_executor(None, heavy_computation)
    return {"result": result}

client = TestClient(app)
print("Sync:  ", client.get("/sync-task").json())
print("Async: ", client.get("/async-task").json())
print("Gen:   ", client.get("/async-gen").text)
print("CPU:   ", client.get("/cpu-task").json())

---
## Module 8 — Middleware

Middleware wraps every request/response. It runs **before** the route handler and **after**.

```
Request → [Middleware 1] → [Middleware 2] → Route Handler
Response ← [Middleware 1] ← [Middleware 2] ← Route Handler
```

Use cases: logging, auth, timing, correlation IDs, rate limiting, CORS

In [ ]:
from fastapi import FastAPI, Request, Response
from fastapi.middleware.cors import CORSMiddleware
from fastapi.testclient import TestClient
import time
import uuid

app = FastAPI()

# --- Timing + Request ID Middleware ---
@app.middleware("http")
async def add_request_metadata(request: Request, call_next):
    request_id = str(uuid.uuid4())[:8]
    start = time.time()

    response: Response = await call_next(request)

    duration = round((time.time() - start) * 1000, 2)
    response.headers["X-Request-ID"] = request_id
    response.headers["X-Response-Time"] = f"{duration}ms"
    print(f"[{request_id}] {request.method} {request.url.path} → {response.status_code} ({duration}ms)")
    return response

# --- CORS Middleware ---
app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000", "https://myapp.com"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"]
)

# --- Simple Rate Limiter Middleware ---
from collections import defaultdict

request_counts: Dict[str, int] = defaultdict(int)
RATE_LIMIT = 100   # per IP

@app.middleware("http")
async def rate_limit(request: Request, call_next):
    client_ip = request.client.host if request.client else "unknown"
    request_counts[client_ip] += 1
    if request_counts[client_ip] > RATE_LIMIT:
        from fastapi.responses import JSONResponse
        return JSONResponse(status_code=429, content={"error": "Rate limit exceeded"})
    return await call_next(request)

@app.get("/ping")
def ping():
    return {"pong": True}

client = TestClient(app)
r = client.get("/ping")
print("Status:        ", r.status_code)
print("Request-ID:    ", r.headers.get("x-request-id"))
print("Response-Time: ", r.headers.get("x-response-time"))

---
## Module 9 — Error Handling

- `HTTPException` — raise standard HTTP errors from any route
- Custom exception classes — define your own domain errors
- Global exception handlers — catch exceptions app-wide and return structured responses
- Always return structured errors in production: `{"error": "...", "code": "...", "detail": "..."}`

In [ ]:
from fastapi import FastAPI, HTTPException, Request
from fastapi.exceptions import RequestValidationError
from fastapi.responses import JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()

# --- Custom Exception Classes ---
class NotFoundException(Exception):
    def __init__(self, resource: str, id: int):
        self.resource = resource
        self.id = id

class PermissionDeniedException(Exception):
    def __init__(self, action: str):
        self.action = action

# --- Global Exception Handlers ---
@app.exception_handler(NotFoundException)
async def not_found_handler(request: Request, exc: NotFoundException):
    return JSONResponse(
        status_code=404,
        content={"error": "not_found", "resource": exc.resource, "id": exc.id}
    )

@app.exception_handler(PermissionDeniedException)
async def permission_handler(request: Request, exc: PermissionDeniedException):
    return JSONResponse(
        status_code=403,
        content={"error": "permission_denied", "action": exc.action}
    )

@app.exception_handler(RequestValidationError)
async def validation_handler(request: Request, exc: RequestValidationError):
    errors = [{"field": e["loc"][-1], "message": e["msg"]} for e in exc.errors()]
    return JSONResponse(status_code=422, content={"error": "validation_failed", "details": errors})

# --- Routes ---
FAKE_DB = {1: "Alice", 2: "Bob"}

@app.get("/users/{user_id}")
def get_user(user_id: int):
    if user_id not in FAKE_DB:
        raise NotFoundException("user", user_id)   # custom exception
    return {"id": user_id, "name": FAKE_DB[user_id]}

@app.delete("/admin/users/{user_id}")
def delete_user(user_id: int, is_admin: bool = False):
    if not is_admin:
        raise PermissionDeniedException("delete_user")
    raise HTTPException(status_code=204)

class Item(BaseModel):
    name: str
    price: float

@app.post("/items")
def create(item: Item):
    return item

client = TestClient(app)
print("Found:       ", client.get("/users/1").json())
print("Not found:   ", client.get("/users/99").json())
print("Permission:  ", client.delete("/admin/users/1").json())
print("Validation:  ", client.post("/items", json={"name": "x", "price": "bad"}).json())

---
## Module 10 — Lifespan Management

Lifespan lets you run code on **startup** and **shutdown** of the app.  
Use it to initialize expensive resources once: DB pools, vector DBs, embedding models, LLM clients.

```python
from contextlib import asynccontextmanager

@asynccontextmanager
async def lifespan(app: FastAPI):
    # --- STARTUP: runs before first request ---
    app.state.db = await init_db()
    app.state.llm = LLMClient()
    yield
    # --- SHUTDOWN: runs after last request ---
    await app.state.db.close()
```

In [ ]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
from contextlib import asynccontextmanager

# Simulated clients
class FakeDBPool:
    def __init__(self): print("  [Startup] DB Pool initialized")
    async def close(self): print("  [Shutdown] DB Pool closed")
    def query(self, q): return f"Result for: {q}"

class FakeVectorDB:
    def __init__(self): print("  [Startup] VectorDB connected")
    def search(self, q): return [f"doc_{i}" for i in range(3)]

class FakeLLMClient:
    def __init__(self): print("  [Startup] LLM Client ready")
    def complete(self, prompt): return "Mock LLM response"

@asynccontextmanager
async def lifespan(app: FastAPI):
    print("=== APP STARTING ===")
    # Initialize resources
    app.state.db = FakeDBPool()
    app.state.vector_db = FakeVectorDB()
    app.state.llm = FakeLLMClient()
    yield
    # Cleanup
    print("=== APP SHUTTING DOWN ===")
    await app.state.db.close()

app = FastAPI(lifespan=lifespan)

@app.get("/query")
def query(q: str, request: Request):
    result = request.app.state.db.query(q)
    return {"query": q, "result": result}

@app.get("/search")
def search(q: str, request: Request):
    docs = request.app.state.vector_db.search(q)
    return {"docs": docs}

@app.post("/generate")
def generate(prompt: str, request: Request):
    resp = request.app.state.llm.complete(prompt)
    return {"response": resp}

client = TestClient(app)   # triggers lifespan startup
print(client.get("/query?q=SELECT * FROM users").json())
print(client.get("/search?q=RAG architecture").json())
print(client.post("/generate?prompt=What is FastAPI?").json())

---
## Module 11 — Background Tasks

`BackgroundTasks` runs functions **after** the response is sent to the client.  
The client doesn't wait — the task runs in the background.

Perfect for: sending emails, generating embeddings, logging, webhook calls, async document processing.

In [ ]:
from fastapi import FastAPI, BackgroundTasks
from fastapi.testclient import TestClient
import time

app = FastAPI()

# Background task functions
def send_email(to: str, subject: str):
    print(f"  [BG] Sending email to {to}: {subject}")
    time.sleep(0.01)   # simulate slow email
    print(f"  [BG] Email sent!")

def generate_embeddings(doc_id: str, text: str):
    print(f"  [BG] Generating embeddings for doc {doc_id}")
    # In production: call embedding model here
    print(f"  [BG] Embeddings stored for {doc_id}")

def log_to_analytics(event: str, user_id: str):
    print(f"  [BG] Analytics: {event} by {user_id}")

@app.post("/register")
def register(email: str, background_tasks: BackgroundTasks):
    # Respond immediately — email is sent in background
    background_tasks.add_task(send_email, email, "Welcome to our platform!")
    background_tasks.add_task(log_to_analytics, "user_registered", email)
    return {"message": "Registered! Check your email."}  # returned immediately

@app.post("/documents/upload")
def upload_doc(doc_id: str, text: str, background_tasks: BackgroundTasks):
    background_tasks.add_task(generate_embeddings, doc_id, text)
    return {"doc_id": doc_id, "status": "uploaded", "embedding": "processing"}

client = TestClient(app)
print(client.post("/register?email=shriman@example.com").json())
print(client.post("/documents/upload?doc_id=doc123&text=Quarterly+report+data").json())

---
## Module 12 — API Architecture

### APIRouter
Breaks your API into modular files. Each router handles one domain (users, items, auth).

### Project Structure (Production)
```
app/
├── main.py               ← FastAPI app + lifespan
├── routers/
│   ├── users.py
│   ├── documents.py
│   └── auth.py
├── schemas/              ← Pydantic models
├── services/             ← Business logic
├── repositories/         ← DB access layer
├── dependencies/         ← Shared Depends()
├── middleware/
└── models/               ← SQLAlchemy models
```

### Versioning
```python
app.include_router(v1_router, prefix="/v1")
app.include_router(v2_router, prefix="/v2")
```

In [ ]:
from fastapi import FastAPI, APIRouter
from fastapi.testclient import TestClient
from pydantic import BaseModel

# --- users router (would be in routers/users.py) ---
users_router = APIRouter(prefix="/users", tags=["Users"])

@users_router.get("/")
def list_users(): return [{"id": 1, "name": "Alice"}, {"id": 2, "name": "Bob"}]

@users_router.get("/{user_id}")
def get_user(user_id: int): return {"id": user_id, "name": f"User {user_id}"}

@users_router.post("/", status_code=201)
def create_user(name: str): return {"id": 99, "name": name}

# --- documents router (would be in routers/documents.py) ---
docs_router = APIRouter(prefix="/documents", tags=["Documents"])

@docs_router.get("/")
def list_docs(): return [{"id": "doc1", "title": "Report Q1"}]

@docs_router.post("/upload")
def upload(title: str): return {"id": "doc2", "title": title, "status": "uploaded"}

# --- Versioning ---
v1_router = APIRouter(prefix="/v1")
v2_router = APIRouter(prefix="/v2")

@v1_router.get("/status")
def v1_status(): return {"version": "v1", "stable": True}

@v2_router.get("/status")
def v2_status(): return {"version": "v2", "features": ["streaming", "agents"]}

# --- Main app ---
app = FastAPI(title="Modular FastAPI")
app.include_router(users_router)
app.include_router(docs_router)
app.include_router(v1_router)
app.include_router(v2_router)

client = TestClient(app)
print("Users:   ", client.get("/users/").json())
print("Docs:    ", client.get("/documents/").json())
print("V1:      ", client.get("/v1/status").json())
print("V2:      ", client.get("/v2/status").json())

---
## Module 13 — Database Integration

FastAPI uses SQLAlchemy (sync or async) for DB access.  
The **Repository Pattern** separates DB logic from route logic.

### Session Management with Depends()
```python
def get_db():
    db = SessionLocal()
    try:
        yield db       # inject into route
    finally:
        db.close()     # always close
```

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from fastapi.testclient import TestClient
from sqlalchemy import create_engine, Column, Integer, String
from sqlalchemy.orm import sessionmaker, DeclarativeBase, Session
from pydantic import BaseModel

# --- SQLAlchemy Setup ---
DATABASE_URL = "sqlite:///./test.db"   # use postgresql://... in production

engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(bind=engine)

class Base(DeclarativeBase): pass

# --- ORM Model ---
class UserModel(Base):
    __tablename__ = "users"
    id = Column(Integer, primary_key=True, index=True)
    username = Column(String, unique=True, index=True)
    email = Column(String)

Base.metadata.create_all(engine)

# --- Pydantic Schemas ---
class UserCreate(BaseModel):
    username: str
    email: str

class UserOut(BaseModel):
    model_config = {"from_attributes": True}  # read from ORM object
    id: int
    username: str
    email: str

# --- Repository ---
class UserRepository:
    def __init__(self, db: Session):
        self.db = db
    def get_all(self): return self.db.query(UserModel).all()
    def get_by_id(self, user_id: int): return self.db.query(UserModel).filter(UserModel.id == user_id).first()
    def create(self, data: UserCreate):
        user = UserModel(**data.model_dump())
        self.db.add(user); self.db.commit(); self.db.refresh(user)
        return user

# --- Dependency ---
def get_db():
    db = SessionLocal()
    try: yield db
    finally: db.close()

app = FastAPI()

@app.post("/users", response_model=UserOut, status_code=201)
def create_user(data: UserCreate, db: Session = Depends(get_db)):
    return UserRepository(db).create(data)

@app.get("/users", response_model=list[UserOut])
def list_users(db: Session = Depends(get_db)):
    return UserRepository(db).get_all()

@app.get("/users/{user_id}", response_model=UserOut)
def get_user(user_id: int, db: Session = Depends(get_db)):
    user = UserRepository(db).get_by_id(user_id)
    if not user: raise HTTPException(404, "User not found")
    return user

client = TestClient(app)
print(client.post("/users", json={"username": "shriman", "email": "s@example.com"}).json())
print(client.post("/users", json={"username": "alice", "email": "a@example.com"}).json())
print(client.get("/users").json())
print(client.get("/users/1").json())

---
## Module 14 — Authentication & Authorization

---

### OAuth2 — What It Is

OAuth2 is an **authorization framework** (not authentication). It defines how a client gets permission to access resources on behalf of a user.

**4 OAuth2 Flows:**

| Flow | Used when |
|---|---|
| **Authorization Code** | Web apps (Google Login, GitHub OAuth) — most secure |
| **Client Credentials** | Service-to-service (no user involved) |
| **Resource Owner Password** | First-party apps (you control client + server) — FastAPI default |
| **Implicit** | Deprecated — don't use |

**OAuth2 Password Flow (FastAPI default):**
```
Client → POST /token {username, password}
Server → validates credentials → returns {access_token, token_type}
Client → GET /users/me (Authorization: Bearer <token>)
Server → validates token → returns user data
```

---

### JWT — JSON Web Token

A JWT is a **self-contained, signed token** — the server doesn't need to look it up in a database to verify it.

**Structure:** `header.payload.signature` (3 parts, base64url-encoded, dot-separated)

```
eyJhbGciOiJIUzI1NiIsInR5cCI6IkpXVCJ9    ← header: algorithm + type
.eyJzdWIiOiJ1c2VyMTIzIiwiZXhwIjoxNjkwfQ  ← payload: claims
.SflKxwRJSMeKKF2QT4fwpMeJf36POk6yJV_adQssw  ← signature: HMAC-SHA256
```

**Common Claims in Payload:**
| Claim | Meaning |
|---|---|
| `sub` | Subject (user ID) |
| `exp` | Expiry timestamp |
| `iat` | Issued At |
| `iss` | Issuer |
| `role` | Custom — user role (admin, viewer) |
| `permissions` | Custom — list of allowed actions |

**How verification works:**
1. Server receives token
2. Decodes header + payload (no secret needed — they're just base64)
3. Recomputes signature using the secret key
4. Compares with the token's signature — if they match → valid
5. Checks `exp` claim — if expired → reject

**Why JWT?**
- Stateless — server doesn't store sessions
- Fast — no DB lookup to verify
- Portable — works across microservices

**Downside:** Can't revoke before expiry → use **short-lived access tokens + refresh tokens**

---

### Access Token vs Refresh Token

| | Access Token | Refresh Token |
|---|---|---|
| Lifetime | 15–60 min | 7–30 days |
| Stored | Memory / header | Secure cookie / DB |
| Used for | Every API call | Only to get new access token |
| Can revoke? | No (short-lived) | Yes (store in DB with revocation list) |

**Flow:**
```
1. Login → get access_token (30min) + refresh_token (7 days)
2. API calls → use access_token in Authorization header
3. access_token expires → use refresh_token to get new access_token
4. refresh_token expires → user must log in again
```

In [ ]:
from jose import jwt, JWTError
from passlib.context import CryptContext
from datetime import datetime, timedelta

# --- Config ---
SECRET_KEY = "your-super-secret-key-change-in-production"  # store in .env!
ALGORITHM = "HS256"
ACCESS_TOKEN_EXPIRE_MINUTES = 30
REFRESH_TOKEN_EXPIRE_DAYS = 7

# --- Password hashing ---
pwd_context = CryptContext(schemes=["bcrypt"], deprecated="auto")

def hash_password(plain: str) -> str:
    return pwd_context.hash(plain)

def verify_password(plain: str, hashed: str) -> bool:
    return pwd_context.verify(plain, hashed)

# --- JWT token creation ---
def create_token(data: dict, expires_delta: timedelta) -> str:
    payload = data.copy()
    payload["exp"] = datetime.utcnow() + expires_delta
    payload["iat"] = datetime.utcnow()
    return jwt.encode(payload, SECRET_KEY, algorithm=ALGORITHM)

def create_access_token(user_id: str, role: str = "viewer") -> str:
    return create_token(
        {"sub": user_id, "role": role, "type": "access"},
        timedelta(minutes=ACCESS_TOKEN_EXPIRE_MINUTES)
    )

def create_refresh_token(user_id: str) -> str:
    return create_token(
        {"sub": user_id, "type": "refresh"},
        timedelta(days=REFRESH_TOKEN_EXPIRE_DAYS)
    )

# --- JWT token verification ---
def decode_token(token: str) -> dict:
    try:
        payload = jwt.decode(token, SECRET_KEY, algorithms=[ALGORITHM])
        return payload
    except JWTError as e:
        raise ValueError(f"Invalid token: {e}")

# --- Demo ---
hashed = hash_password("mysecret123")
print("Hash:    ", hashed)
print("Verify:  ", verify_password("mysecret123", hashed))
print("Wrong:   ", verify_password("wrongpass", hashed))

access = create_access_token("user123", role="admin")
print("\nAccess token:", access[:50] + "...")

decoded = decode_token(access)
print("Decoded:     ", decoded)

refresh = create_refresh_token("user123")
print("\nRefresh token:", refresh[:50] + "...")

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, status
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()

# OAuth2 scheme — tells FastAPI where the token comes from
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")

# Fake user DB
FAKE_USERS = {
    "shriman": {"id": "u1", "username": "shriman", "role": "admin",  "hashed_pw": hash_password("pass123")},
    "alice":   {"id": "u2", "username": "alice",   "role": "viewer", "hashed_pw": hash_password("alice456")},
}

class TokenResponse(BaseModel):
    access_token: str
    refresh_token: str
    token_type: str = "bearer"

class UserOut(BaseModel):
    id: str
    username: str
    role: str

# --- Login endpoint ---
@app.post("/auth/token", response_model=TokenResponse)
def login(form: OAuth2PasswordRequestForm = Depends()):
    user = FAKE_USERS.get(form.username)
    if not user or not verify_password(form.password, user["hashed_pw"]):
        raise HTTPException(status_code=401, detail="Invalid credentials")
    return TokenResponse(
        access_token=create_access_token(user["id"], role=user["role"]),
        refresh_token=create_refresh_token(user["id"])
    )

# --- Get current user dependency ---
def get_current_user(token: str = Depends(oauth2_scheme)) -> dict:
    try:
        payload = decode_token(token)
        if payload.get("type") != "access":
            raise ValueError("Not an access token")
        user_id = payload.get("sub")
        user = next((u for u in FAKE_USERS.values() if u["id"] == user_id), None)
        if not user:
            raise ValueError("User not found")
        return user
    except ValueError as e:
        raise HTTPException(status_code=401, detail=str(e))

@app.get("/auth/me", response_model=UserOut)
def get_me(user: dict = Depends(get_current_user)):
    return user

client = TestClient(app)

# Login
r = client.post("/auth/token", data={"username": "shriman", "password": "pass123"})
tokens = r.json()
print("Login response:", {k: v[:30]+"..." if k.endswith("token") else v for k, v in tokens.items()})

# Use access token
headers = {"Authorization": f"Bearer {tokens['access_token']}"}
me = client.get("/auth/me", headers=headers)
print("Me:            ", me.json())

# Wrong password
r = client.post("/auth/token", data={"username": "shriman", "password": "wrong"})
print("Wrong pass:    ", r.status_code, r.json())

In [ ]:
from fastapi import FastAPI, Depends, HTTPException
from fastapi.security import OAuth2PasswordBearer
from fastapi.testclient import TestClient
from typing import Callable

app = FastAPI()
oauth2_scheme = OAuth2PasswordBearer(tokenUrl="/auth/token")

# Reuse get_current_user from previous cell
def get_current_user(token: str = Depends(oauth2_scheme)) -> dict:
    try:
        payload = decode_token(token)
        user_id = payload.get("sub")
        user = next((u for u in FAKE_USERS.values() if u["id"] == user_id), None)
        if not user: raise ValueError("Not found")
        return user
    except: raise HTTPException(401, "Invalid token")

# --- RBAC: require specific role ---
def require_role(*roles: str) -> Callable:
    def dependency(user: dict = Depends(get_current_user)):
        if user["role"] not in roles:
            raise HTTPException(403, f"Role required: {roles}. You have: {user['role']}")
        return user
    return dependency

@app.get("/admin/stats")
def admin_stats(user: dict = Depends(require_role("admin"))):
    return {"total_users": 1000, "accessed_by": user["username"]}

@app.get("/reports")
def get_reports(user: dict = Depends(require_role("admin", "viewer"))):
    return {"reports": ["Q1", "Q2"], "for": user["username"]}

client = TestClient(app)

# Admin token
admin_r = client.post("/auth/token", data={"username": "shriman", "password": "pass123"})
admin_token = admin_r.json()["access_token"]

# Viewer token
viewer_r = client.post("/auth/token", data={"username": "alice", "password": "alice456"})
viewer_token = viewer_r.json()["access_token"]

admin_h  = {"Authorization": f"Bearer {admin_token}"}
viewer_h = {"Authorization": f"Bearer {viewer_token}"}

print("Admin → /admin/stats: ", client.get("/admin/stats", headers=admin_h).json())
print("Viewer → /admin/stats:", client.get("/admin/stats", headers=viewer_h).status_code)  # 403
print("Viewer → /reports:    ", client.get("/reports", headers=viewer_h).json())

In [ ]:
from fastapi import FastAPI, Security, HTTPException
from fastapi.security import APIKeyHeader
from fastapi.testclient import TestClient

app = FastAPI()

API_KEY_HEADER = APIKeyHeader(name="X-API-Key")

VALID_API_KEYS = {
    "key-prod-abc123": {"client": "service-a", "tier": "premium"},
    "key-dev-xyz789":  {"client": "service-b", "tier": "free"}
}

def verify_api_key(api_key: str = Security(API_KEY_HEADER)):
    client_info = VALID_API_KEYS.get(api_key)
    if not client_info:
        raise HTTPException(status_code=403, detail="Invalid API key")
    return client_info

@app.get("/data")
def get_data(client: dict = Depends(verify_api_key)):
    return {"data": "sensitive info", "accessed_by": client["client"], "tier": client["tier"]}

client_app = TestClient(app)
print("Valid key:   ", client_app.get("/data", headers={"X-API-Key": "key-prod-abc123"}).json())
print("Invalid key: ", client_app.get("/data", headers={"X-API-Key": "bad-key"}).status_code)

---
## Module 16 — Streaming APIs (Critical for GenAI)

### Why Streaming?
- LLMs generate tokens one by one — don't make users wait for the full response
- Large file downloads — don't buffer in memory
- Real-time data feeds (logs, metrics, events)

### SSE (Server-Sent Events)
- One-way: server → client
- Format: `data: {json}\n\n` (double newline = end of event)
- Content-Type: `text/event-stream`
- Client reconnects automatically on disconnect
- Perfect for: LLM streaming, live notifications, progress updates

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
import asyncio
import json

app = FastAPI()

# --- Server-Sent Events ---
@app.get("/events")
async def sse_events():
    async def event_generator():
        events = [
            {"type": "start",    "data": "Stream started"},
            {"type": "progress", "data": "Processing chunk 1"},
            {"type": "progress", "data": "Processing chunk 2"},
            {"type": "done",     "data": "Stream complete"},
        ]
        for event in events:
            yield f"data: {json.dumps(event)}\n\n"   # SSE format: data: ...\n\n
            await asyncio.sleep(0)
    return StreamingResponse(event_generator(), media_type="text/event-stream")

# --- Simulated LLM Token Streaming ---
@app.post("/llm/stream")
async def llm_stream(prompt: str):
    async def token_generator():
        # In production: iterate over anthropic/openai streaming response
        words = f"Here is my response to: {prompt}. FastAPI makes streaming easy with StreamingResponse.".split()
        for word in words:
            token_event = {"token": word + " ", "done": False}
            yield f"data: {json.dumps(token_event)}\n\n"
            await asyncio.sleep(0.01)   # simulate token delay
        yield f"data: {json.dumps({'token': '', 'done': True})}\n\n"
    return StreamingResponse(token_generator(), media_type="text/event-stream")

# --- Progress streaming ---
@app.post("/process")
async def process_with_progress(doc_count: int = 5):
    async def progress():
        for i in range(doc_count):
            yield f"data: {json.dumps({'step': i+1, 'total': doc_count, 'pct': round((i+1)/doc_count*100)})}\n\n"
            await asyncio.sleep(0)
        yield f"data: {json.dumps({'done': True, 'message': 'All processed'})}\n\n"
    return StreamingResponse(progress(), media_type="text/event-stream")

client = TestClient(app)
print("SSE events:")
print(client.get("/events").text)
print("LLM stream:")
print(client.post("/llm/stream?prompt=What+is+RAG").text[:300])

In [ ]:
# Real Claude/Anthropic Streaming Pattern
# Uncomment and use with actual anthropic library

"""
import anthropic
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
import json

app = FastAPI()
client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from env

@app.post("/chat/stream")
async def chat_stream(prompt: str):
    async def token_generator():
        # Stream from Claude
        with client.messages.stream(
            model="claude-sonnet-4-6",
            max_tokens=1000,
            messages=[{"role": "user", "content": prompt}]
        ) as stream:
            for text in stream.text_stream:
                yield f"data: {json.dumps({'token': text, 'done': False})}\\n\\n"
        yield f"data: {json.dumps({'token': '', 'done': True})}\\n\\n"

    return StreamingResponse(token_generator(), media_type="text/event-stream")
"""

# Simulated version for practice:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
import asyncio, json

app = FastAPI()

async def mock_llm_stream(prompt: str):
    """Simulates anthropic/openai streaming chunks"""
    response = f"Based on your question about '{prompt}', here is a detailed answer from the LLM model."
    for char in response:
        yield char
        await asyncio.sleep(0)

@app.post("/chat/stream")
async def chat_stream(prompt: str):
    async def generate():
        async for chunk in mock_llm_stream(prompt):
            yield f"data: {json.dumps({'token': chunk, 'done': False})}\n\n"
        yield f"data: {json.dumps({'token': '', 'done': True})}\n\n"
    return StreamingResponse(generate(), media_type="text/event-stream")

client = TestClient(app)
response_text = client.post("/chat/stream?prompt=What+is+FastAPI").text
# Collect all tokens
tokens = [json.loads(line.replace("data: ", "")) for line in response_text.strip().split("\n\n") if line]
full_response = "".join(t["token"] for t in tokens)
print("Streamed response:", full_response)

---
## Module 17 — WebSockets

WebSockets are **bidirectional** — both client and server can send messages anytime.  
Unlike SSE (server → client only), WebSockets allow full duplex communication.

**Use cases:**
- Real-time chat
- Collaborative editing
- Live agent status updates
- Interactive AI assistants (user sends message, server streams response)

In [ ]:
from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.testclient import TestClient
import json

app = FastAPI()

# --- Basic WebSocket ---
@app.websocket("/ws")
async def websocket_echo(ws: WebSocket):
    await ws.accept()
    try:
        while True:
            data = await ws.receive_text()
            await ws.send_text(f"Echo: {data}")
    except WebSocketDisconnect:
        print("Client disconnected")

# --- AI Chat WebSocket ---
@app.websocket("/ws/chat")
async def ai_chat(ws: WebSocket):
    await ws.accept()
    await ws.send_json({"type": "connected", "message": "AI Assistant ready!"})
    try:
        while True:
            user_msg = await ws.receive_text()
            # Simulate streaming response
            words = f"I understand you asked: {user_msg}. Let me think...".split()
            for word in words:
                await ws.send_json({"type": "token", "token": word + " "})
            await ws.send_json({"type": "done"})
    except WebSocketDisconnect:
        pass

# --- Connection Manager (multi-client) ---
class ConnectionManager:
    def __init__(self):
        self.active: list[WebSocket] = []
    async def connect(self, ws: WebSocket):
        await ws.accept()
        self.active.append(ws)
    def disconnect(self, ws: WebSocket):
        self.active.remove(ws)
    async def broadcast(self, msg: str):
        for ws in self.active:
            await ws.send_text(msg)

manager = ConnectionManager()

@app.websocket("/ws/broadcast/{client_id}")
async def broadcast_ws(ws: WebSocket, client_id: str):
    await manager.connect(ws)
    try:
        while True:
            msg = await ws.receive_text()
            await manager.broadcast(f"[{client_id}]: {msg}")
    except WebSocketDisconnect:
        manager.disconnect(ws)

# Test using TestClient WebSocket
client = TestClient(app)

with client.websocket_connect("/ws") as ws:
    ws.send_text("Hello FastAPI!")
    response = ws.receive_text()
    print("Echo WS:", response)

with client.websocket_connect("/ws/chat") as ws:
    connected_msg = ws.receive_json()
    print("AI WS connected:", connected_msg)
    ws.send_text("What is RAG?")
    tokens = []
    while True:
        msg = ws.receive_json()
        if msg["type"] == "done": break
        tokens.append(msg["token"])
    print("AI response:", "".join(tokens))

---
## Module 18 — Caching with Redis

Caching reduces latency and cost — especially critical for LLM APIs.

**Cache-Aside Pattern:**
1. Check cache for result
2. If hit → return cached
3. If miss → compute, store in cache, return

**LLM response caching** is especially valuable — same prompt = same response, no reason to call the API twice.

In [ ]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient
import hashlib
import json

app = FastAPI()

# Simple in-memory cache (replace with Redis in production)
class MockRedis:
    def __init__(self): self._store = {}
    def get(self, key): return self._store.get(key)
    def set(self, key, value, ex=None): self._store[key] = value
    def delete(self, key): self._store.pop(key, None)

cache = MockRedis()

def get_cache() -> MockRedis:
    return cache

# --- Cache-Aside Pattern ---
def cache_key(prefix: str, **kwargs) -> str:
    raw = json.dumps(kwargs, sort_keys=True)
    return f"{prefix}:{hashlib.md5(raw.encode()).hexdigest()}"

@app.get("/expensive-query")
def expensive_query(q: str, db: MockRedis = Depends(get_cache)):
    key = cache_key("query", q=q)
    cached = db.get(key)
    if cached:
        return {"result": json.loads(cached), "source": "cache"}
    # Simulate expensive operation
    result = {"query": q, "answer": f"Computed result for: {q}"}
    db.set(key, json.dumps(result), ex=300)  # TTL 5 min
    return {"result": result, "source": "computed"}

# --- LLM Response Cache ---
@app.post("/llm/cached")
def llm_cached(prompt: str, model: str = "claude-sonnet-4-6", db: MockRedis = Depends(get_cache)):
    key = cache_key("llm", prompt=prompt, model=model)
    cached = db.get(key)
    if cached:
        return {"response": cached, "cached": True, "cost": 0}
    # In production: response = llm_client.complete(prompt)
    response = f"[Mock LLM response to: {prompt[:40]}]"
    db.set(key, response, ex=3600)  # cache 1 hour
    return {"response": response, "cached": False, "cost": 0.002}

client = TestClient(app)
print("First call:  ", client.get("/expensive-query?q=What+is+RAG").json())
print("Second call: ", client.get("/expensive-query?q=What+is+RAG").json())  # from cache
print("LLM (new):   ", client.post("/llm/cached?prompt=Explain+transformers").json())
print("LLM (cached):", client.post("/llm/cached?prompt=Explain+transformers").json())

---
## Module 19 — Observability

**The three pillars:**
- **Logs** — what happened (structured JSON logs)
- **Metrics** — how much / how fast (Prometheus)
- **Traces** — where time was spent (OpenTelemetry)

**Structured logging** means logging JSON instead of plain text — makes logs searchable in tools like Datadog, Grafana Loki, CloudWatch.

In [ ]:
from fastapi import FastAPI, Request
from fastapi.testclient import TestClient
import logging
import json
import uuid
import time

# --- Structured JSON Logger ---
class JSONFormatter(logging.Formatter):
    def format(self, record):
        log = {
            "timestamp": self.formatTime(record),
            "level": record.levelname,
            "message": record.getMessage(),
            "module": record.module,
        }
        if hasattr(record, "extra"):
            log.update(record.extra)
        return json.dumps(log)

handler = logging.StreamHandler()
handler.setFormatter(JSONFormatter())
logger = logging.getLogger("api")
logger.setLevel(logging.INFO)
logger.handlers = [handler]

app = FastAPI()

# --- Request logging middleware ---
@app.middleware("http")
async def log_requests(request: Request, call_next):
    request_id = str(uuid.uuid4())[:8]
    start = time.time()

    response = await call_next(request)

    duration = round((time.time() - start) * 1000, 2)
    logger.info("request", extra={
        "extra": {
            "request_id": request_id,
            "method": request.method,
            "path": str(request.url.path),
            "status_code": response.status_code,
            "duration_ms": duration
        }
    })
    response.headers["X-Request-ID"] = request_id
    return response

@app.get("/items/{item_id}")
def get_item(item_id: int):
    logger.info(f"Fetching item {item_id}")
    return {"id": item_id, "name": f"Item {item_id}"}

client = TestClient(app)
r = client.get("/items/42")
print("Response:", r.json())
print("Request-ID:", r.headers.get("x-request-id"))

---
## Module 20 — Testing

- `TestClient` — synchronous test client (wraps httpx), no server needed
- **Dependency overrides** — swap real DB/LLM with mocks in tests
- `pytest` — test runner

```python
app.dependency_overrides[get_db] = lambda: FakeDB()
```

In [ ]:
from fastapi import FastAPI, Depends
from fastapi.testclient import TestClient
from pydantic import BaseModel

app = FastAPI()

# --- Real dependency (DB) ---
class RealDB:
    def get_user(self, user_id: int):
        # In production: query actual database
        return {"id": user_id, "name": "From Real DB"}

def get_db() -> RealDB:
    return RealDB()

# --- Real dependency (LLM) ---
class RealLLM:
    def complete(self, prompt: str) -> str:
        # In production: call anthropic/openai
        return f"Real LLM response to: {prompt}"

def get_llm() -> RealLLM:
    return RealLLM()

@app.get("/users/{user_id}")
def get_user(user_id: int, db: RealDB = Depends(get_db)):
    return db.get_user(user_id)

@app.post("/ask")
def ask(prompt: str, llm: RealLLM = Depends(get_llm)):
    return {"response": llm.complete(prompt)}

# --- Mock implementations for testing ---
class MockDB:
    def get_user(self, user_id: int):
        return {"id": user_id, "name": "Mock User"}  # no real DB needed

class MockLLM:
    def complete(self, prompt: str) -> str:
        return "Mocked response"   # deterministic, no API calls

# --- Override dependencies ---
app.dependency_overrides[get_db]  = lambda: MockDB()
app.dependency_overrides[get_llm] = lambda: MockLLM()

client = TestClient(app)

print("User (mock DB): ", client.get("/users/1").json())
print("LLM (mock):     ", client.post("/ask?prompt=hello").json())

# Clear overrides when done (restore real dependencies)
app.dependency_overrides.clear()
print("\nAfter clear — real DB:", client.get("/users/1").json())

---
## Module 21 — FastAPI for GenAI

This is where FastAPI becomes the backbone of your AI system.

| API | Purpose |
|---|---|
| Document Upload API | Accept files, chunk, store |
| Embedding API | Generate + store embeddings |
| Vector Search API | Semantic search over docs |
| RAG API | Retrieve + generate |
| Agent API | Run agent with tools |
| **MCP Server API** | Expose tools to AI agents (Claude, etc.) |
| Streaming LLM API | Token-by-token streaming |
| Human-in-the-Loop API | Pause for human approval |

In [ ]:
# Document Upload + Chunking API
from fastapi import FastAPI, UploadFile, File, BackgroundTasks
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List
import io

app = FastAPI()

# In-memory store (replace with vector DB in production)
doc_store: Dict[str, dict] = {}

def simple_chunk(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """Sliding window chunker"""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

def process_document(doc_id: str, text: str):
    """Background task: chunk + embed (mock)"""
    chunks = simple_chunk(text)
    doc_store[doc_id]["chunks"] = chunks
    doc_store[doc_id]["status"] = "ready"
    print(f"  [BG] Document {doc_id} processed: {len(chunks)} chunks")

@app.post("/documents/upload")
async def upload_document(
    file: UploadFile = File(...),
    background_tasks: BackgroundTasks = BackgroundTasks()
):
    content = await file.read()
    text = content.decode("utf-8")
    doc_id = str(uuid.uuid4())[:8]

    doc_store[doc_id] = {
        "id": doc_id,
        "filename": file.filename,
        "size": len(content),
        "status": "processing",
        "chunks": []
    }
    background_tasks.add_task(process_document, doc_id, text)
    return {"doc_id": doc_id, "status": "processing", "filename": file.filename}

@app.get("/documents/{doc_id}")
def get_doc_status(doc_id: str):
    doc = doc_store.get(doc_id)
    if not doc: raise HTTPException(404, "Document not found")
    return {"id": doc_id, "status": doc["status"], "chunk_count": len(doc["chunks"])}

client = TestClient(app)
large_text = "FastAPI is a modern Python web framework. " * 50  # simulate document
fake_file = io.BytesIO(large_text.encode())
r = client.post("/documents/upload", files={"file": ("report.txt", fake_file, "text/plain")})
print("Upload response:", r.json())

In [ ]:
# RAG API — Retrieve + Generate
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List, Optional
import asyncio, json

app = FastAPI()

class RAGRequest(BaseModel):
    query: str
    top_k: int = 5
    model: str = "claude-sonnet-4-6"
    stream: bool = False

class RAGResponse(BaseModel):
    query: str
    context_docs: List[str]
    answer: str
    sources: List[str]

# Mock vector search (replace with actual vector DB)
def vector_search(query: str, top_k: int) -> List[dict]:
    return [
        {"id": f"doc_{i}", "text": f"Relevant passage {i} about {query}", "score": 0.95 - i*0.05}
        for i in range(top_k)
    ]

# Mock LLM (replace with anthropic/openai)
def llm_generate(query: str, context: str) -> str:
    return f"Based on the provided context, here is the answer to '{query}': [Generated answer using retrieved docs]"

async def llm_stream_generate(query: str, context: str):
    answer = f"Based on context: The answer to '{query}' is derived from the retrieved documents."
    for word in answer.split():
        yield word + " "
        await asyncio.sleep(0)

# Non-streaming RAG
@app.post("/rag", response_model=RAGResponse)
def rag_query(req: RAGRequest):
    docs = vector_search(req.query, req.top_k)
    context = "\n".join(d["text"] for d in docs)
    answer = llm_generate(req.query, context)
    return RAGResponse(
        query=req.query,
        context_docs=[d["text"] for d in docs],
        answer=answer,
        sources=[d["id"] for d in docs]
    )

# Streaming RAG
@app.post("/rag/stream")
async def rag_stream(req: RAGRequest):
    docs = vector_search(req.query, req.top_k)
    context = "\n".join(d["text"] for d in docs)

    async def generate():
        # First send retrieved docs
        yield f"data: {json.dumps({'type': 'sources', 'sources': [d['id'] for d in docs]})}\n\n"
        # Then stream the answer
        async for token in llm_stream_generate(req.query, context):
            yield f"data: {json.dumps({'type': 'token', 'token': token})}\n\n"
        yield f"data: {json.dumps({'type': 'done'})}\n\n"

    return StreamingResponse(generate(), media_type="text/event-stream")

client = TestClient(app)
r = client.post("/rag", json={"query": "What is RAG?", "top_k": 3})
print("RAG response:", json.dumps(r.json(), indent=2))

In [ ]:
# MCP (Model Context Protocol) Server API
# MCP lets AI agents (Claude, etc.) discover and call your tools
# Transport: SSE (Server-Sent Events) or STDIO

from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List, Any
import asyncio, json

app = FastAPI(title="MCP Server", description="Exposes tools to AI agents via MCP protocol")

# --- Tool definitions ---
MCP_TOOLS = [
    {
        "name": "search_documents",
        "description": "Search through the document store using semantic similarity",
        "inputSchema": {
            "type": "object",
            "properties": {
                "query":  {"type": "string", "description": "Search query"},
                "top_k": {"type": "integer", "description": "Number of results", "default": 5}
            },
            "required": ["query"]
        }
    },
    {
        "name": "get_document",
        "description": "Retrieve a specific document by ID",
        "inputSchema": {
            "type": "object",
            "properties": {
                "doc_id": {"type": "string", "description": "Document ID"}
            },
            "required": ["doc_id"]
        }
    },
    {
        "name": "run_sql_query",
        "description": "Execute a read-only SQL query against the database",
        "inputSchema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "SQL SELECT query"}
            },
            "required": ["query"]
        }
    }
]

# --- Tool executor ---
def execute_tool(name: str, arguments: dict) -> Any:
    if name == "search_documents":
        query = arguments["query"]
        top_k = arguments.get("top_k", 5)
        return [{"id": f"doc_{i}", "text": f"Result {i} for: {query}", "score": 0.9-i*0.1}
                for i in range(top_k)]
    elif name == "get_document":
        doc_id = arguments["doc_id"]
        return {"id": doc_id, "title": f"Document {doc_id}", "content": "Full document content here..."}
    elif name == "run_sql_query":
        return [{"row": 1, "data": "sample"}, {"row": 2, "data": "sample2"}]
    else:
        raise ValueError(f"Unknown tool: {name}")

# --- MCP Endpoints ---

# List available tools
@app.get("/mcp/tools")
def list_tools():
    return {"tools": MCP_TOOLS}

class ToolCallRequest(BaseModel):
    name: str
    arguments: dict

# Call a tool
@app.post("/mcp/tools/call")
def call_tool(req: ToolCallRequest):
    try:
        result = execute_tool(req.name, req.arguments)
        return {
            "content": [{"type": "text", "text": json.dumps(result, indent=2)}],
            "isError": False
        }
    except ValueError as e:
        return {"content": [{"type": "text", "text": str(e)}], "isError": True}

# SSE transport — used by Claude to connect to MCP server
@app.get("/mcp/sse")
async def mcp_sse_transport():
    async def event_stream():
        # Send server capabilities
        init = {
            "jsonrpc": "2.0",
            "method": "initialize",
            "params": {
                "protocolVersion": "2024-11-05",
                "capabilities": {"tools": {}},
                "serverInfo": {"name": "my-mcp-server", "version": "1.0.0"}
            }
        }
        yield f"data: {json.dumps(init)}\n\n"
        # Keep alive
        for _ in range(3):
            yield ": keepalive\n\n"
            await asyncio.sleep(0)

    return StreamingResponse(event_stream(), media_type="text/event-stream")

client = TestClient(app)
print("MCP Tools:", json.dumps(client.get("/mcp/tools").json()["tools"][0], indent=2))
print("\nTool call:")
r = client.post("/mcp/tools/call", json={"name": "search_documents", "arguments": {"query": "RAG architecture", "top_k": 2}})
print(json.dumps(r.json(), indent=2))

In [ ]:
# Agent API — Run an agent with tools + track state
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel
from typing import List, Optional
import asyncio, json, uuid

app = FastAPI()

class AgentRunRequest(BaseModel):
    task: str
    tools: List[str] = ["search_documents", "run_sql_query"]
    max_steps: int = 10
    stream: bool = True

class AgentStep(BaseModel):
    step: int
    thought: str
    action: Optional[str] = None
    action_input: Optional[dict] = None
    observation: Optional[str] = None

# In-memory run store
agent_runs: Dict[str, dict] = {}

async def run_agent(run_id: str, task: str, tools: List[str], max_steps: int):
    """Simulated ReAct agent loop"""
    steps = []
    for step in range(1, max_steps + 1):
        thought = f"Step {step}: Analyzing task '{task}'"
        action = tools[step % len(tools)] if step < max_steps else None
        observation = f"Result from {action}: relevant data found" if action else "Task complete"

        step_data = AgentStep(
            step=step, thought=thought,
            action=action,
            action_input={"query": task} if action else None,
            observation=observation
        )
        steps.append(step_data.model_dump())
        agent_runs[run_id]["steps"] = steps
        await asyncio.sleep(0)

        if step >= 3:  # stop after 3 steps for demo
            break

    agent_runs[run_id]["status"] = "completed"
    agent_runs[run_id]["final_answer"] = f"After {len(steps)} steps, the answer to '{task}' is: [Agent answer]"

@app.post("/agent/run")
async def run_agent_stream(req: AgentRunRequest):
    run_id = str(uuid.uuid4())[:8]
    agent_runs[run_id] = {"status": "running", "steps": [], "final_answer": None}

    if req.stream:
        async def generate():
            yield f"data: {json.dumps({'type': 'start', 'run_id': run_id})}\n\n"
            await run_agent(run_id, req.task, req.tools, req.max_steps)
            for step in agent_runs[run_id]["steps"]:
                yield f"data: {json.dumps({'type': 'step', **step})}\n\n"
            yield f"data: {json.dumps({'type': 'done', 'answer': agent_runs[run_id]['final_answer']})}\n\n"
        return StreamingResponse(generate(), media_type="text/event-stream")

    await run_agent(run_id, req.task, req.tools, req.max_steps)
    return {"run_id": run_id, **agent_runs[run_id]}

@app.get("/agent/run/{run_id}")
def get_run(run_id: str):
    run = agent_runs.get(run_id)
    if not run: raise HTTPException(404, "Run not found")
    return {"run_id": run_id, **run}

client = TestClient(app)
r = client.post("/agent/run", json={"task": "Find all Q1 reports and summarize", "stream": False})
print(json.dumps(r.json(), indent=2))

---
## Module 22 — Deployment

### Running in Production

```bash
# Development
uvicorn main:app --reload

# Production (multiple workers)
gunicorn main:app -w 4 -k uvicorn.workers.UvicornWorker --bind 0.0.0.0:8000
```

### Docker
```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install -r requirements.txt
COPY . .
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
```

### docker-compose.yml
```yaml
services:
  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - DATABASE_URL=postgresql://user:pass@db:5432/mydb
      - REDIS_URL=redis://redis:6379
    depends_on:
      - db
      - redis
  db:
    image: postgres:15
    environment:
      POSTGRES_PASSWORD: pass
  redis:
    image: redis:7
```

In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
import time

app = FastAPI()

# App state tracking
app_start_time = time.time()
db_healthy = True   # would be real DB ping in production

# --- Health check (liveness) --- is the app alive?
@app.get("/health", tags=["Ops"])
def health_check():
    return {
        "status": "healthy",
        "uptime_seconds": round(time.time() - app_start_time, 1),
        "timestamp": time.time()
    }

# --- Readiness check (readiness) --- is the app ready to serve traffic?
@app.get("/ready", tags=["Ops"])
def readiness_check():
    checks = {
        "database": db_healthy,
        "cache": True,           # mock — check Redis ping in production
        "vector_db": True        # mock — check VectorDB in production
    }
    all_healthy = all(checks.values())
    return {
        "ready": all_healthy,
        "checks": checks
    }

# --- Metrics endpoint (for Prometheus scraping) ---
REQUEST_COUNT = 0

@app.middleware("http")
async def count_requests(request, call_next):
    global REQUEST_COUNT
    REQUEST_COUNT += 1
    return await call_next(request)

@app.get("/metrics", tags=["Ops"])
def metrics():
    return {
        "requests_total": REQUEST_COUNT,
        "uptime_seconds": round(time.time() - app_start_time, 1)
        # In production: expose Prometheus format with prometheus-fastapi-instrumentator
    }

client = TestClient(app)
print("Health:  ", client.get("/health").json())
print("Readiness:", client.get("/ready").json())
client.get("/health")  # increment counter
print("Metrics: ", client.get("/metrics").json())

---
## Interview Priority Tiers

### Tier 1 — Must Know (practice until automatic)
| Topic | Key thing to demonstrate |
|---|---|
| HTTP Methods | GET/POST/PUT/PATCH/DELETE with TestClient |
| Pydantic | `Field()`, `@field_validator`, `response_model` |
| `Depends()` | Auth dependency, DB session, chaining |
| APIRouter | Modular structure, prefixes, tags |
| Async/Await | When to use each, never block event loop |
| Lifespan | Init DB/LLM/VectorDB on startup |
| Middleware | Request logging, timing, CORS |
| SQLAlchemy | Session with Depends, repository pattern |
| JWT | Create token, decode, protect routes |
| `StreamingResponse` | Generator function, LLM token streaming |

### Tier 2 — Strongly Recommended
- SSE (`text/event-stream`)
- WebSockets + ConnectionManager
- BackgroundTasks (embedding generation)
- Redis caching (LLM response cache)
- Testing with dependency overrides
- OpenTelemetry / Langfuse tracing

### Tier 3 — Senior-Level
- Async SQLAlchemy + Alembic migrations
- Kubernetes + Gunicorn worker tuning
- Connection pooling strategies
- MCP Server implementation
- Multi-agent FastAPI architecture
- FastAPI performance profiling